In [ ]:
import json
import logging

import re
import subprocess
import sys
import tempfile
from dataclasses import dataclass
from typing import Any, Dict, List, Optional
from pathlib import Path

import joblib
import numpy as np
from scipy import sparse

import requests

logger = logging.getLogger(__name__)
logger.setLevel(logging.INFO)

## Agents

In [ ]:
def _ensure_utf8_output() -> None:
    """Force UTF-8 stdout/stderr to avoid codec errors on GBK consoles."""

    for stream in (sys.stdout, sys.stderr):
        reconfig = getattr(stream, "reconfigure", None)
        if callable(reconfig):  # pragma: no cover - environment dependent
            try:
                reconfig(encoding="utf-8", errors="replace")
            except Exception:  # noqa: BLE001 - best effort guard
                pass

_ensure_utf8_output()

@dataclass
class SolverResult:
    code: str
    approach: str
    assumptions: List[str]
    complexity_claim: Dict[str, str]
    changed_from_last: str

@dataclass
class CriticResult:
    passed: bool
    failure_type: str
    notes: str
    complexity_class: str
    complexity_evidence: List[str]
    suggested_fix: str
    test_summary: Dict[str, Any]

@dataclass
class IterationTrace:
    iteration: int
    retrieval: List[Dict[str, Any]]
    solver: SolverResult
    critic: CriticResult

@dataclass
class Guide:
    guide_title: str
    final_summary: str
    steps: List[Dict[str, Any]]
    pitfalls: List[str]
    final_complexity: Dict[str, str]

class LLMClient:
    def __init__(self, provider: str, api_key: Optional[str] = None, model: Optional[str] = None):
        self.provider = provider
        self.api_key = api_key
        self.model = model or ("deepseek-chat" if provider.lower() == "deepseek" else "gpt-4o-mini")
        self.mock_mode = api_key is None or api_key.strip() == ""

    def complete(self, prompt: str) -> str:
        if self.mock_mode:
            logger.warning("No API key provided; falling back to mock LLM output")
            # Provide deterministic minimal JSON to keep pipeline running
            return "MOCK"
        provider = self.provider.lower()
        if provider == "deepseek":
            base_url = "https://api.deepseek.com"
        else:
            base_url = "https://api.openai.com"

        url = f"{base_url}/v1/chat/completions"
        headers = {
            "Authorization": f"Bearer {self.api_key}",
            "Content-Type": "application/json",
        }
        payload = {
            "model": self.model,
            "messages": [{"role": "user", "content": prompt}],
            "temperature": 0.2,
        }
        try:
            resp = requests.post(url, headers=headers, json=payload, timeout=30)
            resp.raise_for_status()
            data = resp.json()
            return data["choices"][0]["message"]["content"]
        except Exception as exc:  # pragma: no cover - network/HTTP handling
            raise RuntimeError(f"LLM request failed: {exc}") from exc

prompts = ["You are SolverAgent. Write a full Python program for the problem. Use retrieved hints as guidance only. Output strict JSON with keys code, approach, assumptions, complexity_claim, changed_from_last."]

class SolverAgent:
    def __init__(self, llm: LLMClient):
        self.llm = llm

    def generate(self, *, question: str, tests: Dict[str, List[str]], retrieved: List[Dict[str, Any]], starter_code: Optional[str], prev_fix: Optional[str], iteration: int, format_prompt: str = prompts[0]) -> SolverResult:
        retrieval_text = json.dumps(retrieved, ensure_ascii=False, indent=2)
        prompt = format_prompt + f"Problem:\n{question}" + \
            f"\n\nTests:{tests}" if tests else "\n\nTests: You are not given test cases. The formats of test cases are explicitly given in the desciption of the problem." + \
            f"\n\nStarter code:{starter_code}" if starter_code else "\n\nStarter code: No starter codes are given. You should design and write your own starter code." + \
            f"\n\nRetrieved hints:{retrieval_text}" if retrieval_text else "\n\nRetrieved hints: No hints are provided. Make sure your algorithm and code implementation is applicable later in most hidden test cases." + \
            f"\n\nPrevious fix request:{prev_fix}" if prev_fix else "\n\nPrevious fix request: None."
        raw = self.llm.complete(prompt)
        if raw == "MOCK":
            code = self._mock_code(tests)
            return SolverResult(
                code=code,
                approach="Mock fallback program that echoes expected output.",
                assumptions=["Using mock LLM output"],
                complexity_claim={"time": "O(1)", "space": "O(1)"},
                changed_from_last="(mock mode)" if iteration > 0 else "",
            )
        try:
            data = json.loads(raw)
        except json.JSONDecodeError:
            raise ValueError(f"SolverAgent did not return valid JSON: {raw}")
        complexity_claim = data.get("complexity_claim", {})
        if not isinstance(complexity_claim, dict):
            complexity_claim = {}
        assumptions = data.get("assumptions", [])
        if not isinstance(assumptions, list):
            assumptions = []
        return SolverResult(
            code=data.get("code", ""),
            approach=data.get("approach", ""),
            assumptions=assumptions,
            complexity_claim=complexity_claim,
            changed_from_last=data.get("changed_from_last", ""),
        )

    def _mock_code(self, tests: Dict[str, List[str]]) -> str:
        inputs = tests.get("inputs") or []
        outputs = tests.get("outputs") or []
        pairs = {i: o for i, o in zip(inputs, outputs)}
        mapping_lines = ",".join(
            [
                f"{json.dumps(k)}: {json.dumps((v or '').strip())}"
                for k, v in pairs.items()
            ]
        )
        return (
            "import sys\n"
            "# Mock solution produced because no API key was provided.\n"
            f"mapping = {{{mapping_lines}}}\n"
            "data = sys.stdin.read()\n"
            "if data in mapping:\n"
            "    print(mapping[data])\n"
            "else:\n"
            "    print(mapping.get(data.strip(), ''))\n"
        )


class CriticAgent:
    def __init__(self, timeout: float = 2.0):
        self.timeout = timeout

    def evaluate(self, code: str, tests: Dict[str, List[str]], question: str) -> CriticResult:
        complexity_class, evidence = estimate_complexity(code)
        passed, summary, failure_type, notes = self._run_tests(code, tests)
        allowed, extra_note = complexity_gate(question, complexity_class)
        if not allowed:
            failure_type = "COMPLEXITY"
            notes = (notes + "; " if notes else "") + extra_note
            passed = False
        test_summary = summary
        suggested_fix = self._suggest_fix(failure_type, notes)
        return CriticResult(
            passed=passed,
            failure_type=failure_type,
            notes=notes,
            complexity_class=complexity_class,
            complexity_evidence=evidence,
            suggested_fix=suggested_fix,
            test_summary=test_summary,
        )

    def _run_tests(self, code: str, tests: Dict[str, List[str]]):
        inputs = tests.get("inputs") or []
        outputs = tests.get("outputs") or []
        num_passed = 0
        first_failure = None
        failure_type = "WA"
        notes = ""
        for idx, (inp, expected) in enumerate(zip(inputs, outputs)):
            result = self._run_single(code, inp, expected)
            if result[0]:
                num_passed += 1
            else:
                failure_type, notes, got = result[1], result[2], result[3]
                first_failure = {"idx": idx, "expected": expected, "got": got}
                break
        passed = num_passed == len(inputs)
        summary = {
            "num_tests": len(inputs),
            "num_passed": num_passed,
            "first_failure": first_failure,
        }
        if passed:
            failure_type = "WA"
            notes = ""
        return passed, summary, failure_type if not passed else "WA", notes

    def _run_single(self, code: str, input_str: str, expected_output: str):
        with tempfile.NamedTemporaryFile("w", suffix=".py", delete=False) as tmp:
            tmp.write(code)
            tmp_path = tmp.name
        try:
            proc = subprocess.run(
                ["python", tmp_path],
                input=input_str,
                text=True,
                capture_output=True,
                timeout=self.timeout,
            )
        except subprocess.TimeoutExpired:
            return False, "TLE", "Time limit exceeded", ""
        stdout = normalize_output(proc.stdout)
        expected_norm = normalize_output(expected_output)
        if proc.returncode != 0:
            return False, "RE", proc.stderr[:400], stdout
        if stdout.strip() != expected_norm.strip():
            return False, "WA", "Wrong answer", stdout
        return True, "", "", stdout

    def _suggest_fix(self, failure_type: str, notes: str) -> str:
        if failure_type == "WA":
            return "Review logic against sample tests and ensure outputs match exactly."
        if failure_type == "RE":
            return f"Fix runtime error: {notes[:120]}"
        if failure_type == "TLE":
            return "Optimize loops/recursion to avoid timeouts."
        if failure_type == "COMPLEXITY":
            return "Replace nested loops with linear approach based on constraints."
        return "Investigate issues from critic feedback."


class GuiderAgent:
    def synthesize(self, traces: List[IterationTrace]) -> Guide:
        steps = []
        for t in traces:
            steps.append(
                {
                    "iteration": t.iteration,
                    "what_failed_or_risk": t.critic.failure_type if not t.critic.passed else "OK",
                    "what_we_changed": t.solver.changed_from_last or "Initial attempt",
                    "evidence": t.critic.notes or json.dumps(t.critic.test_summary),
                    "complexity_before_after": {
                        "before": t.solver.complexity_claim.get("time", "unknown"),
                        "after": t.critic.complexity_class,
                    },
                }
            )
        pitfalls = ["Keep outputs normalized (trim trailing spaces)", "Watch complexity gates for large N"]
        final_complexity = {
            "time": traces[-1].critic.complexity_class if traces else "unknown",
            "space": traces[-1].solver.complexity_claim.get("space", "unknown") if traces else "unknown",
        }
        return Guide(
            guide_title="How the solution evolved",
            final_summary="Concise walkthrough of solver and critic iterations.",
            steps=steps,
            pitfalls=pitfalls,
            final_complexity=final_complexity,
        )


def parse_problem_payload(payload_text: str) -> Dict[str, Any]:
    try:
        payload = json.loads(payload_text)
    except json.JSONDecodeError as exc:
        raise ValueError(f"Invalid JSON payload: {exc}")
    if not isinstance(payload, dict):
        raise ValueError("Payload must be a JSON object")
    for key in ["question", "input_output"]:
        if key not in payload:
            raise ValueError(f"Missing required field: {key}")
    return payload


def parse_tests(input_output_raw: Any) -> Dict[str, List[str]]:
    if isinstance(input_output_raw, str):
        try:
            parsed = json.loads(input_output_raw)
        except json.JSONDecodeError as exc:
            raise ValueError(f"input_output string must be valid JSON: {exc}")
        # Some payloads may be doubly string-encoded (string containing JSON string)
        if isinstance(parsed, str):
            try:
                parsed = json.loads(parsed)
            except json.JSONDecodeError as exc:
                raise ValueError(
                    "input_output string contained nested JSON that could not be parsed: "
                    f"{exc}"
                )
    elif isinstance(input_output_raw, dict):
        parsed = input_output_raw
    else:
        raise ValueError("input_output must be string or dict")
    if not isinstance(parsed, dict):
        raise ValueError("input_output must decode to an object/dict")
    if "inputs" not in parsed or "outputs" not in parsed:
        raise ValueError("input_output missing inputs/outputs")
    inputs = parsed.get("inputs")
    outputs = parsed.get("outputs")
    if not isinstance(inputs, list) or not isinstance(outputs, list):
        raise ValueError("inputs/outputs must be lists")
    if len(inputs) != len(outputs):
        raise ValueError("inputs and outputs must have the same length")
    return {"inputs": inputs, "outputs": outputs}


def normalize_output(text: str) -> str:
    return "\n".join(line.rstrip() for line in (text or "").replace("\r\n", "\n").split("\n")).strip()


def estimate_complexity(code: str):
    lines = [ln.strip() for ln in code.splitlines() if ln.strip() and not ln.strip().startswith("#")]
    loop_depth = 0
    max_depth = 0
    for ln in lines:
        if re.match(r"(for |while )", ln):
            loop_depth += 1
            max_depth = max(max_depth, loop_depth)
        if ln.endswith(":") is False:
            loop_depth = max(loop_depth - 1, 0)
    if max_depth >= 3:
        cls = "O(N^3)"
    elif max_depth == 2:
        cls = "O(N^2)"
    elif max_depth == 1:
        cls = "O(N)"
    else:
        cls = "O(1)"
    evidence = [f"Detected nested loop depth={max_depth}"]
    if "recursion" in code.lower():
        evidence.append("recursion keyword spotted")
    return cls, evidence


def complexity_gate(question: str, complexity_class: str):
    note = ""
    constraints_large = re.search(r"1e5|10\^5|100000", question)
    constraints_mid = re.search(r"1e4|10\^4|10000", question)
    if constraints_large:
        if complexity_class in {"O(N^2)", "O(N^3)", "O(2^N)", "O(N!)"}:
            return False, "Complexity too high for N>=1e5"
    elif constraints_mid:
        if complexity_class in {"O(N^2)", "O(N^3)", "O(2^N)", "O(N!)"}:
            return False, "Complexity too high for N around 1e4"
    else:
        if complexity_class in {"O(N^3)", "O(2^N)", "O(N!)"}:
            return False, "Rejected by default complexity gate"
    return True, note


def run_pipeline(
    payload_text: str,
    provider: str,
    api_key: Optional[str],
    retrieved: List[Dict[str, Any]],
    max_iters: int = 3,
    timeout: float = 2.0,
):
    payload = parse_problem_payload(payload_text)
    tests = parse_tests(payload.get("input_output"))
    question = payload.get("question", "")
    starter_code = payload.get("starter_code")
    llm = LLMClient(provider, api_key)
    solver = SolverAgent(llm)
    critic = CriticAgent(timeout=timeout)
    guider = GuiderAgent()

    traces: List[IterationTrace] = []
    prev_fix = None
    for i in range(max_iters):
        solver_result = solver.generate(
            question=question,
            tests=tests,
            retrieved=retrieved,
            starter_code=starter_code,
            prev_fix=prev_fix,
            iteration=i,
        )
        critic_result = critic.evaluate(solver_result.code, tests, question)
        traces.append(
            IterationTrace(
                iteration=i + 1,
                retrieval=retrieved,
                solver=solver_result,
                critic=critic_result,
            )
        )
        if critic_result.passed and critic_result.failure_type != "COMPLEXITY":
            break
        prev_fix = critic_result.suggested_fix

    guider_output = guider.synthesize(traces)
    final_pass = traces[-1].critic.passed and traces[-1].critic.failure_type != "COMPLEXITY"
    final_code = traces[-1].solver.code
    return {
        "decision": "PASS" if final_pass else "REJECT",
        "final_code": final_code,
        "guide": guider_output,
        "traces": traces,
        "tests": tests,
    }


## RAG

In [4]:
@dataclass
class RetrievalResult:
    problem_id: str
    score: float
    question_snippet: str
    difficulty: Optional[str] = None
    url: Optional[str] = None
    starter_code: Optional[str] = None
    solution_snippet: Optional[str] = None


class ProblemDatabase:
    def __init__(self, artifact_dir: str = "data/reference_db"):
        self.artifact_dir = Path(artifact_dir)
        self.records: List[dict] = []
        self.word_vectorizer = None
        self.char_vectorizer = None
        self.matrix: Optional[sparse.csr_matrix] = None

    def is_ready(self) -> bool:
        required = [
            self.artifact_dir / "apps.jsonl",
            self.artifact_dir / "tfidf_matrix.npz",
            self.artifact_dir / "tfidf_vectorizer_words.joblib",
            self.artifact_dir / "tfidf_vectorizer_chars.joblib",
        ]
        return all(p.exists() for p in required)

    def load(self):
        if not self.is_ready():
            raise FileNotFoundError(
                "Reference DB artifacts missing. Run scripts/build_reference_db.py and scripts/build_tfidf_index.py"
            )
        self.records = []
        with (self.artifact_dir / "apps.jsonl").open("r", encoding="utf-8") as f:
            for line in f:
                self.records.append(json.loads(line))
        self.word_vectorizer = joblib.load(self.artifact_dir / "tfidf_vectorizer_words.joblib")
        self.char_vectorizer = joblib.load(self.artifact_dir / "tfidf_vectorizer_chars.joblib")
        self.matrix = sparse.load_npz(self.artifact_dir / "tfidf_matrix.npz")
        logger.info("Loaded %d records into RAG DB", len(self.records))

    def _build_query_vec(self, query: str):
        word_vec = self.word_vectorizer.transform([query])
        char_vec = self.char_vectorizer.transform([query])
        return sparse.hstack([word_vec, char_vec]).tocsr()

    def search(self, query: str, k: int = 3) -> List[RetrievalResult]:
        if self.matrix is None:
            raise RuntimeError("RAG DB not loaded. Call load() first.")
        query_vec = self._build_query_vec(query)
        sims = self.matrix.dot(query_vec.T).toarray().ravel()
        top_idx = np.argsort(-sims)[:k]
        results: List[RetrievalResult] = []
        for idx in top_idx:
            rec = self.records[int(idx)]
            score = float(sims[int(idx)])
            results.append(
                RetrievalResult(
                    problem_id=str(rec.get("problem_id")),
                    score=score,
                    question_snippet=_trim_text(rec.get("question", "")),
                    difficulty=rec.get("difficulty"),
                    url=rec.get("url"),
                    starter_code=_trim_text(rec.get("starter_code") or "", 400),
                    solution_snippet=_trim_text(_first_solution(rec.get("solutions")), 400),
                )
            )
        return results


def _trim_text(text: str, limit: int = 600) -> str:
    if not text:
        return ""
    if len(text) <= limit:
        return text
    return text[: limit - 3] + "..."


def _first_solution(raw) -> str:
    if raw is None:
        return ""
    if isinstance(raw, list) and raw:
        return str(raw[0])
    if isinstance(raw, str):
        return raw
    return str(raw)


## Evaluate

In [ ]:
import time
from collections import Counter, defaultdict

from tqdm import tqdm

DATA_PATH = Path("data/reference_db/apps.jsonl")
OUTPUT_PATH = Path("results/eval_results.jsonl")
SUMMARY_PATH = Path("results/summary.json")

PROVIDER = "DeepSeek"        # or "OpenAI"
API_KEY = ""               # set via env or here
MAX_ITERS = 3
RAG_K = 3
TIMEOUT = 2.0


def load_apps_data(path: Path) -> List[Dict[str, Any]]:
    records = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            records.append(json.loads(line))
    return records


def evaluate():
    OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

    # ---- Load data ----
    problems = load_apps_data(DATA_PATH)
    print(f"Loaded {len(problems)} APPS problems")

    # ---- Load RAG DB ----
    db = ProblemDatabase()
    if not db.is_ready():
        raise RuntimeError("RAG DB not built. Build reference DB first.")
    db.load()

    stats = Counter()
    failure_types = Counter()
    complexity_blocks = 0
    iter_hist = Counter()

    t_start = time.time()
    
    i = 0

    with OUTPUT_PATH.open("w", encoding="utf-8") as fout:
        for idx, payload in enumerate(tqdm(problems, desc="Evaluating")):
            i += 1
            if i > 10:
                break
            question = payload.get("question", "")
            print(question)
            retrieved = db.search(question, k=RAG_K)
            retrieved_dicts = [r.__dict__ for r in retrieved]

            try:
                result = run_pipeline(
                    payload_text=json.dumps(payload),
                    provider=PROVIDER,
                    api_key=API_KEY,
                    retrieved=retrieved_dicts,
                    max_iters=MAX_ITERS,
                    timeout=TIMEOUT,
                )

            except Exception as exc:
                record = {
                    "problem_id": payload.get("problem_id", idx),
                    "status": "ERROR",
                    "error": str(exc),
                }
                fout.write(json.dumps(record) + "\n")
                stats["ERROR"] += 1
                continue

            decision = result["decision"]
            traces = result.get("traces", [])

            num_iters = len(traces)
            iter_hist[num_iters] += 1

            final_critic = traces[-1].critic
            failure = final_critic.failure_type if decision != "PASS" else "PASS"

            if failure == "COMPLEXITY":
                complexity_blocks += 1

            failure_types[failure] += 1
            stats[decision] += 1

            record = {
                "problem_id": payload.get("problem_id", idx),
                "decision": decision,
                "num_iterations": num_iters,
                "failure_type": failure,
                "final_complexity": final_critic.complexity_class,
                "retrieval_scores": [r["score"] for r in retrieved_dicts],
                "used_rag": len(retrieved_dicts),
            }

            fout.write(json.dumps(record) + "\n")

    elapsed = time.time() - t_start

    summary = {
        "total": len(problems),
        "passed": stats["PASS"],
        "rejected": stats["REJECT"],
        "error": stats["ERROR"],
        "pass_rate": stats["PASS"] / max(len(problems), 1),
        "avg_time_per_problem_sec": elapsed / max(len(problems), 1),
        "failure_types": dict(failure_types),
        "iterations_distribution": dict(iter_hist),
        "complexity_blocks": complexity_blocks,
        "config": {
            "provider": PROVIDER,
            "max_iters": MAX_ITERS,
            "rag_k": RAG_K,
            "timeout": TIMEOUT,
        },
    }

    with SUMMARY_PATH.open("w", encoding="utf-8") as f:
        json.dump(summary, f, indent=2)

    print("=== Evaluation Complete ===")
    print(json.dumps(summary, indent=2))


if __name__ == "__main__":
    evaluate()

Loaded 500 APPS problems


INFO:__main__:Loaded 500 records into RAG DB
Evaluating:   0%|          | 0/500 [00:00<?, ?it/s]

Polycarp has $n$ different binary words. A word called binary if it contains only characters '0' and '1'. For example, these words are binary: "0001", "11", "0" and "0011100".

Polycarp wants to offer his set of $n$ binary words to play a game "words". In this game, players name words and each next word (starting from the second) must start with the last character of the previous word. The first word can be any. For example, these sequence of words can be named during the game: "0101", "1", "10", "00", "00001".

Word reversal is the operation of reversing the order of the characters. For example, the word "0111" after the reversal becomes "1110", the word "11010" after the reversal becomes "01011".

Probably, Polycarp has such a set of words that there is no way to put them in the order correspondent to the game rules. In this situation, he wants to reverse some words from his set so that:  the final set of $n$ words still contains different words (i.e. all words are unique);  there is

Evaluating:   0%|          | 1/500 [00:31<4:18:01, 31.03s/it]

Mikhail walks on a Cartesian plane. He starts at the point $(0, 0)$, and in one move he can go to any of eight adjacent points. For example, if Mikhail is currently at the point $(0, 0)$, he can go to any of the following points in one move:   $(1, 0)$;  $(1, 1)$;  $(0, 1)$;  $(-1, 1)$;  $(-1, 0)$;  $(-1, -1)$;  $(0, -1)$;  $(1, -1)$. 

If Mikhail goes from the point $(x1, y1)$ to the point $(x2, y2)$ in one move, and $x1 \ne x2$ and $y1 \ne y2$, then such a move is called a diagonal move.

Mikhail has $q$ queries. For the $i$-th query Mikhail's target is to go to the point $(n_i, m_i)$ from the point $(0, 0)$ in exactly $k_i$ moves. Among all possible movements he want to choose one with the maximum number of diagonal moves. Your task is to find the maximum number of diagonal moves or find that it is impossible to go from the point $(0, 0)$ to the point $(n_i, m_i)$ in $k_i$ moves.

Note that Mikhail can visit any point any number of times (even the destination point!).


-----Input--

...

You like playing chess tournaments online.

In your last tournament you played $n$ games. For the sake of this problem, each chess game is either won or lost (no draws). When you lose a game you get $0$ points. When you win you get $1$ or $2$ points: if you have won also the previous game you get $2$ points, otherwise you get $1$ point. If you win the very first game of the tournament you get $1$ point (since there is not a "previous game").

The outcomes of the $n$ games are represented by a string $s$ of length $n$: the $i$-th character of $s$ is W if you have won the $i$-th game, while it is L if you have lost the $i$-th game.

After the tournament, you notice a bug on the website that allows you to change the outcome of at most $k$ of your games (meaning that at most $k$ times you can change some symbol L to W, or W to L). Since your only goal is to improve your chess rating, you decide to cheat and use the bug.

Compute the maximum score you can get by cheating in the optimal way.

Evaluating:   2%|▏         | 9/500 [04:35<4:09:27, 30.48s/it]

Alice and Bob play a game. They have a binary string $s$ (a string such that each character in it is either $0$ or $1$). Alice moves first, then Bob, then Alice again, and so on.

During their move, the player can choose any number (not less than one) of consecutive equal characters in $s$ and delete them.

For example, if the string is $10110$, there are $6$ possible moves (deleted characters are bold):  $\textbf{1}0110 \to 0110$;  $1\textbf{0}110 \to 1110$;  $10\textbf{1}10 \to 1010$;  $101\textbf{1}0 \to 1010$;  $10\textbf{11}0 \to 100$;  $1011\textbf{0} \to 1011$. 

After the characters are removed, the characters to the left and to the right of the removed block become adjacent. I. e. the following sequence of moves is valid: $10\textbf{11}0 \to 1\textbf{00} \to 1$.

The game ends when the string becomes empty, and the score of each player is the number of $1$-characters deleted by them.

Each player wants to maximize their score. Calculate the resulting score of Alice.


-----Inp

Evaluating:   2%|▏         | 10/500 [05:06<4:10:15, 30.64s/it]

=== Evaluation Complete ===
{
  "total": 500,
  "passed": 0,
  "rejected": 0,
  "error": 10,
  "pass_rate": 0.0,
  "avg_time_per_problem_sec": 0.6128737545013427,
  "failure_types": {},
  "iterations_distribution": {},
  "complexity_blocks": 0,
  "config": {
    "provider": "DeepSeek",
    "max_iters": 3,
    "rag_k": 3,
    "timeout": 2.0
  }
}


In [ ]:
import argparse
import datetime as _dt
import json
import os
import random
import sys
import time
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple
from dataclasses import asdict, is_dataclass
from pathlib import Path
from typing import Any
from tqdm import tqdm

from concurrent.futures import ThreadPoolExecutor, as_completed

# Ensure repo root is on sys.path (so we can import agents.py / rag_engine.py)
REPO_ROOT = os.path.abspath(os.path.join(os.path.dirname(__file__), ".."))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

from datasets import load_dataset  # type: ignore

from agents import LLMClient, SolverAgent, CriticAgent, parse_problem_payload, parse_tests, run_pipeline
from rag_engine import ProblemDatabase, RetrievalResult

def _to_jsonable(x: Any):
    if x is None or isinstance(x, (str, int, float, bool)):
        return x
    if isinstance(x, Path):
        return str(x)
    if isinstance(x, dict):
        return {str(k): _to_jsonable(v) for k, v in x.items()}
    if isinstance(x, (list, tuple)):
        return [_to_jsonable(v) for v in x]
    if isinstance(x, set):
        return [_to_jsonable(v) for v in x]
    if is_dataclass(x):
        return _to_jsonable(asdict(x))
    if hasattr(x, "__dict__"):
        return _to_jsonable(vars(x))
    return str(x)

def _ensure_dir(p: Path) -> None:
    p.mkdir(parents=True, exist_ok=True)


def _timestamp_tag() -> str:
    return _dt.datetime.now().strftime("%Y%m%d_%H%M%S")


def _make_payload_from_row(row: Dict[str, Any]) -> Dict[str, Any]:
    # Keep the same shape that app.py expects.
    return {
        "question": row.get("question", ""),
        "input_output": row.get("input_output", {}),
        "solutions": row.get("solutions", []),
        "starter_code": row.get("starter_code", ""),
        "meta": {
            "problem_id": row.get("problem_id"),
            "difficulty": row.get("difficulty"),
            "url": row.get("url"),
            "apps_config": row.get("apps_config", row.get("apps_config_name")),
            "apps_split": row.get("apps_split"),
        },
    }


def _select_eval_problems(
    seed: int,
    n_eval: int,
    configs: List[str],
    split: str,
    cache_dir: Optional[str],
) -> List[Dict[str, Any]]:
    """
    Deterministically sample n_eval problems from Hugging Face APPS across given configs.
    """
    rng = random.Random(seed)
    chosen: List[Dict[str, Any]] = []
    chosen_ids: set[str] = set()

    # Shuffle configs deterministically to avoid bias
    configs_shuffled = configs[:]
    rng.shuffle(configs_shuffled)

    # Round-robin pull from each config until we have enough
    per_config_seed_base = seed * 1000 + 7
    while len(chosen) < n_eval:
        progress = False
        for ci, cfg in enumerate(configs_shuffled):
            if len(chosen) >= n_eval:
                break
            ds = load_dataset(
                "codeparrot/apps",
                cfg,
                split=split,
                cache_dir=cache_dir,
                trust_remote_code=True,
            )

            # Use HF shuffle for deterministic ordering within this config
            ds_shuf = ds.shuffle(seed=per_config_seed_base + ci)
            # Scan until we find one usable item (stop early for speed)
            for row in ds_shuf:
                pid = row.get("problem_id")
                url = row.get("url")
                key = (str(url).strip() if url else f"{split}:{pid}")

                # If key is empty or already chosen, skip
                if not key or key in chosen_ids:
                    continue

                rec = dict(row)
                rec["apps_config"] = cfg
                rec["apps_split"] = split
                chosen.append(rec)
                chosen_ids.add(key)
                progress = True
                break

        if not progress:
            raise RuntimeError(
                "Could not find enough evaluation problems excluding reference ids. "
                "Try a different --seed, different --eval_split, or reduce --eval_n."
            )

    return chosen


def _retrieve(db: Optional[ProblemDatabase], question: str, rag_k: int) -> List[RetrievalResult]:
    if db is None or rag_k <= 0:
        return []
    return db.search(question, k=rag_k)


def eval_baseline_single(
    payload: Dict[str, Any],
    provider: str,
    api_key: Optional[str],
    retrieved: List[RetrievalResult],
    timeout: float,
) -> Dict[str, Any]:
    llm = LLMClient(provider=provider, api_key=api_key)
    solver = SolverAgent(llm)
    critic = CriticAgent(timeout=timeout)

    # IMPORTANT: parse_tests expects payload["input_output"], not the full payload
    try:
        tests = parse_tests(payload.get("input_output"))
    except Exception as e:
        # Don't crash the whole batch; record and continue
        return {
            "mode": "baseline_single",
            "passed": False,
            "gate_allowed": False,
            "complexity_estimate": None,
            "failure_type": "payload_parse_error",
            "first_failure": None,
            "final_code": "",
            "notes": f"parse_tests failed: {e}",
        }

    res = solver.generate(
        question=payload.get("question", ""),
        tests=tests,
        starter_code=payload.get("starter_code", ""),
        retrieved=[r.__dict__ for r in retrieved],
        prev_fix=None,
        iteration=0,
    )
    cres = critic.evaluate(
        question=payload.get("question", ""),
        code=res.code,
        tests=tests,
    )
    return {
        "mode": "baseline_single",
        "passed": bool(cres.passed),
        "gate_allowed": bool(getattr(cres, "gate_allowed", True)),
        "complexity_estimate": getattr(cres, "complexity", None),
        "failure_type": (
            cres.diagnosis.get("failure_type")
            if isinstance(getattr(cres, "diagnosis", None), dict)
            else getattr(cres, "failure_type", None)
        ),
        "first_failure": getattr(cres, "first_failure", None),
        "final_code": res.code,
    }

def eval_baseline_repeated(
    payload: Dict[str, Any],
    provider: str,
    api_key: Optional[str],
    retrieved: List[RetrievalResult],
    timeout: float,
    repeat_n: int,
    seed: int,
) -> Dict[str, Any]:
    llm = LLMClient(provider=provider, api_key=api_key)
    solver = SolverAgent(llm)
    critic = CriticAgent(timeout=timeout)

    try:
        tests = parse_tests(payload.get("input_output"))
    except Exception as e:
        return {
            "mode": "baseline_repeated",
            "passed": False,
            "gate_allowed": False,
            "attempts_used": 0,
            "repeat_n": repeat_n,
            "failure_type_counts": {"payload_parse_error": 1},
            "first_failure": None,
            "final_code": "",
            "complexity_estimate": None,
            "notes": f"parse_tests failed: {e}",
        }


    failures: List[str] = []
    attempts_used = 0
    best_code = ""
    first_failure = None
    gate_allowed_best = False
    complexity_best = None

    for i in range(repeat_n):
        attempts_used += 1
        res = solver.generate(
            question=payload.get("question", ""),
            tests=tests,
            starter_code=payload.get("starter_code", ""),
            retrieved=[r.__dict__ for r in retrieved],
            prev_fix=None,  # critical: independent attempts (no feedback)
            iteration=i + 1,  # small prompt variation
        )
        cres = critic.evaluate(
            question=payload.get("question", ""),
            code=res.code,
            tests=tests,
        )
        ftype = None
        if isinstance(getattr(cres, "diagnosis", None), dict):
            ftype = cres.diagnosis.get("failure_type")
        else:
            ftype = getattr(cres, "failure_type", None)
        if not ftype:
            ftype = "unknown"
        if first_failure is None:
            first_failure = getattr(cres, "first_failure", None)

        if bool(cres.passed) and bool(getattr(cres, "gate_allowed", True)):
            best_code = res.code
            gate_allowed_best = True
            complexity_best = getattr(cres, "complexity", None)
            return {
                "mode": "baseline_repeated",
                "passed": True,
                "gate_allowed": True,
                "attempts_used": attempts_used,
                "repeat_n": repeat_n,
                "failure_type_counts": _count_list(failures),
                "first_failure": first_failure,
                "final_code": best_code,
                "complexity_estimate": complexity_best,
            }

        failures.append(ftype)

    # none passed
    return {
        "mode": "baseline_repeated",
        "passed": False,
        "gate_allowed": False,
        "attempts_used": attempts_used,
        "repeat_n": repeat_n,
        "failure_type_counts": _count_list(failures),
        "first_failure": first_failure,
        "final_code": best_code,
        "complexity_estimate": complexity_best,
    }


def eval_agent(
    payload: Dict[str, Any],
    provider: str,
    api_key: Optional[str],
    retrieved: List[RetrievalResult],
    max_iters: int,
    timeout: float,
) -> Dict[str, Any]:
    out = run_pipeline(
        payload_text=json.dumps(payload, ensure_ascii=False),
        provider=provider,
        api_key=api_key,
        retrieved=[r.__dict__ for r in retrieved],
        max_iters=max_iters,
        timeout=timeout,
    )

    decision_raw = out.get("decision")
    traces = out.get("traces") or []

    # derive iters_used if not present
    iters_used = out.get("iters_used")
    if iters_used is None and isinstance(traces, list):
        iters_used = len(traces)

    def _extract_failure_type_from_traces(ts):
        if not isinstance(ts, list) or not ts:
            return None
        last = ts[-1]
        critic = getattr(last, "critic", None)
        if critic is None and isinstance(last, dict):
            critic = last.get("critic")
        if critic is None:
            return None
        # critic may be a dataclass-like object or dict
        if isinstance(critic, dict):
            # prefer diagnosis.failure_type if present
            diag = critic.get("diagnosis")
            if isinstance(diag, dict) and diag.get("failure_type"):
                return diag.get("failure_type")
            return critic.get("failure_type")
        return getattr(critic, "failure_type", None)

    # default outputs
    passed = False
    gate_allowed = True
    failure_type = None

    # Case A: decision is the original string "PASS"/"REJECT"
    if isinstance(decision_raw, str):
        passed = decision_raw.strip().upper() == "PASS"
        gate_allowed = True
        if not passed:
            failure_type = _extract_failure_type_from_traces(traces) or "unknown"

    # Case B: if you later change run_pipeline to return a dict decision, we still support it
    elif isinstance(decision_raw, dict):
        passed = bool(decision_raw.get("passed", False))
        gate_allowed = bool(decision_raw.get("gate_allowed", True))
        diagnosis = decision_raw.get("diagnosis", None)
        if isinstance(diagnosis, dict):
            failure_type = diagnosis.get("failure_type")

    else:
        # unexpected format
        passed = False
        gate_allowed = True
        failure_type = "bad_decision_format"

    return {
        "mode": "agent",
        "passed": passed,
        "gate_allowed": gate_allowed,
        "iters_used": iters_used,
        "failure_type": failure_type,
        "final_code": out.get("final_code"),
        "guide": out.get("guide"),
        "traces": traces,
    }


def _count_list(xs: List[str]) -> Dict[str, int]:
    d: Dict[str, int] = {}
    for x in xs:
        d[x] = d.get(x, 0) + 1
    return d


def _summarize(records: List[Dict[str, Any]], mode: str) -> Dict[str, Any]:
    n = len(records)
    success = 0
    failures: Dict[str, int] = {}
    gate_rejects = 0

    iters_or_attempts: List[int] = []

    for r in records:
        ok = bool(r.get("passed")) and bool(r.get("gate_allowed", True))
        if ok:
            success += 1
            if mode == "agent" and r.get("iters_used") is not None:
                iters_or_attempts.append(int(r["iters_used"]))
            if mode == "baseline_repeated" and r.get("attempts_used") is not None:
                iters_or_attempts.append(int(r["attempts_used"]))
        else:
            ft = r.get("failure_type")
            if ft is None and isinstance(r.get("failure_type_counts"), dict):
                # for repeated: use the most common failure type in attempts
                counts = r["failure_type_counts"]
                if counts:
                    ft = max(counts.items(), key=lambda kv: kv[1])[0]
            if not ft:
                ft = "unknown"
            failures[ft] = failures.get(ft, 0) + 1

        if r.get("passed") and not r.get("gate_allowed", True):
            gate_rejects += 1

    success_rate = success / n if n else 0.0
    avg_steps = sum(iters_or_attempts) / len(iters_or_attempts) if iters_or_attempts else None

    return {
        "mode": mode,
        "n": n,
        "success": success,
        "success_rate": success_rate,
        "gate_rejects": gate_rejects,
        "failure_type_counts": failures,
        "avg_success_steps": avg_steps,  # iters for agent, attempts for repeated
    }


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--out_dir", type=str, default="results", help="Root directory to write results")
    ap.add_argument("--seed", type=int, default=42)
    ap.add_argument("--rag_ks", type=str, default="0,3", help="Comma-separated rag_k values to run, e.g. 0,3,5")
    ap.add_argument("--modes", type=str, default="baseline_single,baseline_repeated,agent",
                    help="Comma-separated: baseline_single, baseline_repeated, agent")
    ap.add_argument("--repeat_n", type=int, default=5, help="N for baseline_repeated (Pass@N best-of-N)")
    ap.add_argument("--max_iters", type=int, default=3, help="Max iters for agent (Pass@K)")
    ap.add_argument("--timeout", type=float, default=2.0)
    ap.add_argument("--provider", type=str, default="DeepSeek")
    ap.add_argument("--api_key", type=str, default=None)
    ap.add_argument("--workers", type=int, default=12,
                help="ThreadPool workers for parallel per-problem eval (I/O bound LLM calls). Use 1 to disable.")


    # Evaluation sampling from HF APPS (leakage-free)
    ap.add_argument("--eval_n", type=int, default=10, help="Number of evaluation problems to sample from APPS each run")
    ap.add_argument("--eval_configs", type=str, default="introductory,interview,competition",
                    help="Comma-separated APPS configs to sample evaluation problems from")
    ap.add_argument("--eval_split", type=str, default="test", help="HF split to sample evaluation problems from (test recommended)")
    ap.add_argument("--cache_dir", type=str, default=None)

    # Reference DB directory for RAG + loaded_ids
    ap.add_argument("--ref_db_dir", type=str, default="data/reference_db")

    args = ap.parse_args()

    out_root = Path(args.out_dir)
    run_dir = out_root / _timestamp_tag()
    _ensure_dir(run_dir)

    modes = [m.strip() for m in args.modes.split(",") if m.strip()]
    rag_ks = [int(x.strip()) for x in args.rag_ks.split(",") if x.strip() != ""]
    eval_configs = [c.strip() for c in args.eval_configs.split(",") if c.strip()]

    ref_db_dir = Path(args.ref_db_dir)

    # Sample evaluation problems from HF (no exclusion needed if you eval on a different split/dataset)
    eval_rows = _select_eval_problems(
        seed=args.seed,
        n_eval=args.eval_n,
        configs=eval_configs,
        split=args.eval_split,
        cache_dir=args.cache_dir,
    )

    # Persist eval ids for reproducibility
    eval_keys = []
    for r in eval_rows:
        url = r.get("url")
        pid = r.get("problem_id")
        split = r.get("apps_split", args.eval_split)
        key = (str(url).strip() if url else f"{split}:{pid}")
        eval_keys.append(key)

    (run_dir / "eval_problem_keys.json").write_text(json.dumps(eval_keys, indent=2), encoding="utf-8")

    # Lazy-load RAG db only if needed.
    db: Optional[ProblemDatabase] = None
    if any(k > 0 for k in rag_ks):
        db = ProblemDatabase(str(ref_db_dir))
        if not db.is_ready():
            raise RuntimeError(
                "RAG artifacts missing (tfidf_matrix.npz + vectorizers). "
                "Build them first via scripts/build_reference_db.py and scripts/build_tfidf_index.py, "
                "or run with --rag_ks 0."
            )
        db.load()

    # Run experiments
    for rag_k in rag_ks:
        for mode in modes:
            records: List[Dict[str, Any]] = []
            t0 = time.time()

            def _process_one(idx_row: Tuple[int, Dict[str, Any]]) -> Dict[str, Any]:
                idx, row = idx_row
                payload = _make_payload_from_row(row)
                question = payload.get("question", "")

                retrieved = _retrieve(db, question, rag_k)
                rec: Dict[str, Any] = {
                    "eval_index": idx,
                    "problem_id": payload.get("meta", {}).get("problem_id"),
                    "difficulty": payload.get("meta", {}).get("difficulty"),
                    "apps_config": payload.get("meta", {}).get("apps_config"),
                    "apps_split": payload.get("meta", {}).get("apps_split"),
                    "rag_k": rag_k,
                }

                start = time.time()
                try:
                    if mode == "baseline_single":
                        res = eval_baseline_single(
                            payload=payload,
                            provider=args.provider,
                            api_key=args.api_key,
                            retrieved=retrieved,
                            timeout=args.timeout,
                        )
                        rec.update(res)
                    elif mode == "baseline_repeated":
                        res = eval_baseline_repeated(
                            payload=payload,
                            provider=args.provider,
                            api_key=args.api_key,
                            retrieved=retrieved,
                            timeout=args.timeout,
                            repeat_n=args.repeat_n,
                            seed=args.seed,
                        )
                        rec.update(res)
                    elif mode == "agent":
                        res = eval_agent(
                            payload=payload,
                            provider=args.provider,
                            api_key=args.api_key,
                            retrieved=retrieved,
                            max_iters=args.max_iters,
                            timeout=args.timeout,
                        )
                        rec.update(res)
                    else:
                        raise ValueError(f"Unknown mode: {mode}")
                except Exception as e:
                    rec.update({
                        "mode": mode,
                        "passed": False,
                        "gate_allowed": False,
                        "failure_type": "exception",
                        "exception": repr(e),
                    })

                rec["runtime_sec"] = time.time() - start
                return rec

            items = list(enumerate(eval_rows))
            workers = max(1, int(getattr(args, "workers", 1)))

            if workers == 1:
                iterator = items
                if tqdm is not None:
                    iterator = tqdm(iterator, total=len(items), desc=f"{mode}/rag{rag_k}", leave=False)
                for item in iterator:
                    records.append(_process_one(item))
            else:
                with ThreadPoolExecutor(max_workers=workers) as ex:
                    futures = [ex.submit(_process_one, item) for item in items]
                    iterator = as_completed(futures)
                    if tqdm is not None:
                        iterator = tqdm(iterator, total=len(futures), desc=f"{mode}/rag{rag_k}", leave=False)
                    for fut in iterator:
                        records.append(fut.result())

            records.sort(key=lambda r: int(r.get("eval_index", 0)))

            # Write JSONL + summary
            tag = f"{mode}_ragk{rag_k}"
            jsonl_out = run_dir / f"{tag}.jsonl"
            with jsonl_out.open("w", encoding="utf-8") as f:
                for r in records:
                    f.write(json.dumps(_to_jsonable(r), ensure_ascii=False) + "\n")

            summary = _summarize(records, mode)
            if mode == "baseline_single":
                summary["pass_at"] = 1
            elif mode == "baseline_repeated":
                summary["pass_at"] = args.repeat_n
            elif mode == "agent":
                summary["pass_at"] = args.max_iters
            summary["rag_k"] = rag_k
            summary["seed"] = args.seed
            summary["eval_n"] = len(records)
            summary["eval_split"] = args.eval_split
            summary["eval_configs"] = eval_configs
            summary["total_runtime_sec"] = time.time() - t0

            summary_out = run_dir / f"{tag}_summary.json"
            summary_out.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")

            tqdm.write(f"[OK] {tag}: success_rate={summary.get('success_rate'):.3f} -> {jsonl_out} | {summary_out}")

    tqdm.write(f"[DONE] Results written to: {run_dir}")


if __name__ == "__main__":
    main()


## APP

In [ ]:
import streamlit as st

logging.basicConfig(level=logging.INFO)

st.set_page_config(page_title="APPS RAG Multi-Agent Demo", layout="wide")


def main():
    st.title("RAG + Multi-Agent APPS Solver")

    db = ProblemDatabase()
    db_ready = db.is_ready()
    if db_ready:
        db.load()
    else:
        st.warning(
            "Reference DB missing. Build it with:\n"
            "python scripts/build_reference_db.py --split test --limit 500\n"
            "python scripts/build_tfidf_index.py"
        )

    tabs = st.tabs(["Problem & Run", "War Room / Agent Debate", "RAG & Analytics"])

    with tabs[0]:
        st.subheader("Paste APPS Problem JSON")
        sample_text = st.session_state.get("last_payload", "")
        payload_text = st.text_area("Problem JSON", sample_text, height=220)

        provider = st.selectbox("Provider", ["DeepSeek", "OpenAI"], index=0)
        api_key = st.text_input("API Key", type="password")
        max_iters = st.number_input("Max iterations", min_value=0, max_value=6, value=3)
        rag_k = st.number_input("RAG top-k", min_value=0, max_value=10, value=3)
        timeout = st.number_input("Timeout per test (sec)", min_value=0.5, max_value=60.0, value=2.0, step=0.5)

        run_btn = st.button("Run Full Pipeline", type="primary")

        if run_btn:
            if not db_ready:
                st.error("RAG DB not built. Build artifacts before running.")
            else:
                try:
                    payload = parse_problem_payload(payload_text)
                except Exception as exc:  # noqa: BLE001
                    st.error(f"Payload parsing error: {exc}")
                    st.stop()
                st.session_state["last_payload"] = payload_text
                if rag_k > 0:
                    retrieved = db.search(payload.get("question", ""), k=int(rag_k))
                    retrieved_dicts = [r.__dict__ for r in retrieved]
                try:
                    results = run_pipeline(
                        payload_text=json.dumps(payload),
                        provider=provider,
                        api_key=api_key,
                        retrieved=retrieved_dicts,
                        max_iters=int(max_iters),
                        timeout=float(timeout),
                    )
                except Exception as exc:  # noqa: BLE001
                    st.error(f"Pipeline failed: {exc}")
                    st.stop()
                st.session_state["run_results"] = results
                st.session_state["retrieved"] = retrieved_dicts

        if "run_results" in st.session_state:
            res = st.session_state["run_results"]
            st.success(f"Final Decision: {res['decision']}")
            st.code(res.get("final_code", ""), language="python")
            guide = res.get("guide")
            if guide:
                st.write("### Teacher-Style Guide")
                st.markdown(render_guide_markdown(guide))

    with tabs[1]:
        st.subheader("War Room")
        traces: List = []
        if "run_results" in st.session_state:
            traces = st.session_state["run_results"].get("traces", [])
        if not traces:
            st.info("Run the pipeline to see agent debate.")
        else:
            for t in traces:
                with st.expander(f"Iteration {t.iteration}"):
                    st.write("#### Retrieval")
                    st.json(t.retrieval)
                    st.write("#### Solver code")
                    st.code(t.solver.code, language="python")
                    st.write("Approach:", t.solver.approach)
                    st.write("#### Critic feedback (summary)")
                    st.markdown(render_critic_summary(t.critic))
                    st.write("Raw Critic JSON")
                    st.json(t.critic.__dict__)
                    if t.solver.changed_from_last:
                        st.write("Changed from last:", t.solver.changed_from_last)

    with tabs[2]:
        st.subheader("RAG & Analytics")
        st.write("DB ready:", db_ready)
        if db_ready:
            st.write("Indexed records:", len(db.records))
        if "retrieved" in st.session_state:
            st.write("Last retrieval set:")
            st.json(st.session_state["retrieved"])
        if "run_results" in st.session_state:
            traces = st.session_state["run_results"].get("traces", [])
            if traces:
                complexity_series = {t.iteration: complexity_to_numeric(t.critic.complexity_class) for t in traces}
                st.line_chart(complexity_series)


def complexity_to_numeric(cls: str) -> int:
    order = {
        "O(1)": 1,
        "O(logN)": 2,
        "O(N)": 3,
        "O(NlogN)": 4,
        "O(N^2)": 5,
        "O(N^3)": 6,
        "O(2^N)": 7,
        "O(N!)": 8,
    }
    return order.get(cls, 0)


def render_guide_markdown(guide: Any) -> str:
    g = guide.__dict__ if hasattr(guide, "__dict__") else guide
    title = g.get("guide_title", "Guide")
    summary = g.get("final_summary", "")
    steps = g.get("steps", []) or []
    pitfalls = g.get("pitfalls", []) or []
    final_complexity = g.get("final_complexity", {}) or {}

    parts = [f"#### {title}", summary or ""]
    if steps:
        parts.append("**Iteration Recap**")
        for step in steps:
            parts.append(
                "- Iteration {iteration}: issue/risk — {issue}. Change — {change}. Evidence — {evidence}. Complexity {before} → {after}.".format(
                    iteration=step.get("iteration", "?"),
                    issue=step.get("what_failed_or_risk", ""),
                    change=step.get("what_we_changed", ""),
                    evidence=step.get("evidence", ""),
                    before=step.get("complexity_before_after", {}).get("before", "unknown"),
                    after=step.get("complexity_before_after", {}).get("after", "unknown"),
                )
            )
    if pitfalls:
        parts.append("**Pitfalls & Reminders**")
        for p in pitfalls:
            parts.append(f"- {p}")
    if final_complexity:
        parts.append(
            "**Final Complexity**: time {time}, space {space}".format(
                time=final_complexity.get("time", "unknown"),
                space=final_complexity.get("space", "unknown"),
            )
        )
    return "\n\n".join(parts)


def render_critic_summary(critic: Any) -> str:
    c = critic.__dict__ if hasattr(critic, "__dict__") else critic
    status = "passed" if c.get("passed") else "failed"
    failure = c.get("failure_type", "?")
    notes = c.get("notes", "")
    suggestion = c.get("suggested_fix", "")
    test_summary = c.get("test_summary", {}) or {}
    num_tests = test_summary.get("num_tests", 0)
    num_passed = test_summary.get("num_passed", 0)
    first_failure = test_summary.get("first_failure")

    human_summary = "The critic {status} the solution (type: {failure}).".format(
        status="approved" if c.get("passed") else "rejected", failure=failure
    )

    lines = [human_summary, f"Tests passed: {num_passed}/{num_tests}."]
    if first_failure:
        lines.append(
            "First failing case #{idx}: expected `{exp}` but got `{got}`.".format(
                idx=first_failure.get("idx"),
                exp=str(first_failure.get("expected", "")),
                got=str(first_failure.get("got", "")),
            )
        )
    if notes:
        lines.append(f"Reasoning: {notes}")
    if suggestion:
        lines.append(f"Suggested next change: {suggestion}")
    lines.append(
        "Complexity gate: {cls}. Evidence: {evidence}".format(
            cls=c.get("complexity_class", "unknown"),
            evidence="; ".join(c.get("complexity_evidence", []) or []),
        )
    )
    return "\n".join(lines)


if __name__ == "__main__":
    main()


2025-12-18 04:01:00.415 
  command:

    streamlit run d:\Anaconda3\envs\COMS4995-032AML\lib\site-packages\ipykernel_launcher.py [ARGUMENTS]
INFO:__main__:Loaded 500 records into RAG DB
2025-12-18 04:01:02.348 Session state does not function when running a script without `streamlit run`
